In [6]:
import os
import datetime

import geopandas as gpd
import pandas as pd
import gtfs_kit

In [18]:
DATAPATH = 'C:/Users/jmuir/source/urban-mobility-simulation/data/infrastructure'
GTFS_IN = os.path.join(DATAPATH, 'GTFS.zip')
TRANSIT_OUT = os.path.join(DATAPATH, 'transit_timetable.csv')
TRANSIT_SHAPES_OUT = os.path.join(DATAPATH, 'transit_shapes.csv')

SCHEMA = ['stop_id', 'stop_lat', 'stop_lon', 'departure_time', 'dep_next_day', 'shape_dist_traveled',
          'route_id', 'service_id', 'trip_id', 'trip_short_name', 'shape_id', 'direction_id', 'stop_sequence',
          'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

# Bus 200 = 17350_700
# Bus 246 = 17362_700
# Bus M11 =

ROUTE = '17350_700' # which route to restrict to
WEEK = ['20210809', '20210810', '20210811', '20210812', '20210813', '20210814', '20210815'] # exemplary week for route timetable

In [19]:
# Load GTFS feed
gtfs = gtfs_kit.read_feed(GTFS_IN, 'm')

# ...and restrict to route
gtfs = gtfs.restrict_to_routes([ROUTE])

# ...then clean
gtfs = gtfs.clean()

# and add optional, hence non-existent (shame on you BVG) shape_dist_travelled fields for distance computations
gtfs = gtfs.append_dist_to_shapes()
gtfs = gtfs.append_dist_to_stop_times()

C:\Users\jmuir\miniconda3\envs\urban-mobility-simulation\lib\site-packages\gtfs_kit\cleaners.py:46: FutureWarning: The default value of regex will change from True to False in a future version.
  f[column] = f[column].str.strip().str.replace(r"\s+", "_")


In [20]:
# build route timetable
timetable = gtfs.build_route_timetable(ROUTE, WEEK)
timetable = timetable.merge(gtfs.stops, on='stop_id')
timetable = timetable.merge(gtfs.calendar, on='service_id')
timetable['route_id'] = ROUTE

# add next day flag on >24:00
timetable['dep_next_day'] = timetable['departure_time'] >= '24:00:00'

# clean up schema
# check if all attributes of SCHEMA are present in the feed, else create an artificial NaN column
for attr in SCHEMA:
    if attr not in list(timetable):
        timetable[attr] = pd.NA
        print('Attribute', attr,
              'was not found in the feed. Inserted empty column. This might lead to unexpected behaviour!')

droplist = list(set(timetable) ^ set(SCHEMA))  # xor operator with two sets yields the difference - then convert that to list
timetable.drop(droplist, inplace=True, axis=1)
# re-arrange columns
timetable = timetable[SCHEMA]

# and clean up duplicated
timetable = timetable[~timetable.duplicated()]

# save both timetable and shapes
timetable.to_csv(TRANSIT_OUT, index=False)
gtfs.shapes.to_csv(TRANSIT_SHAPES_OUT, index=False)

In [21]:
gtfs.validate()

,type,message,table,rows
0,error,Invalid route_type; maybe has extra space char...,routes,[771]
1,warning,Unrecognized column platform_code,stops,[]
2,warning,Unrecognized column from_route_id,transfers,[]
3,warning,Unrecognized column to_route_id,transfers,[]
4,warning,Unrecognized column from_trip_id,transfers,[]
5,warning,Unrecognized column to_trip_id,transfers,[]
